In [4]:
import os
import glob
import cv2
import numpy as np
import pandas as pd
from skimage.feature import graycomatrix, graycoprops

# === ПАПКИ ===
base_dir = r"D:\diplom\dataset_segment"
images_dir = os.path.join(base_dir, "images")
masks_dir = os.path.join(base_dir, "masks")

In [5]:
def load_pair(image_path, masks_dir):
    """Загрузить изображение (gray) и соответствующую маску."""
    fname = os.path.basename(image_path)
    mask_path = os.path.join(masks_dir, fname)
    if not os.path.exists(mask_path):
        return None, None

    # читаем в градациях серого
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    if img is None or mask is None:
        return None, None

    return img, mask

def get_object_background_masks(mask):
    """Получить бинарные маски объекта и фона."""
    # считаем, что маска > 0 — ооцит
    obj = mask > 0
    bg = ~obj
    return obj, bg

1. Градиент на границе и контраст объект/фон

In [6]:
def compute_boundary_gradient_and_contrast(img, mask):
    """Вернуть:
    - mean_grad_boundary, std_grad_boundary
    - contrast_C (объект/фон)
    """
    img_float = img.astype(np.float32)

    # градиенты по Sobel
    gx = cv2.Sobel(img_float, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(img_float, cv2.CV_32F, 0, 1, ksize=3)
    grad_mag = np.sqrt(gx**2 + gy**2)

    # бинарная маска ооцита
    obj_mask, bg_mask = get_object_background_masks(mask.astype(np.uint8))

    # граница: XOR маски и эрозии
    obj_uint = obj_mask.astype(np.uint8)
    kernel = np.ones((3, 3), np.uint8)
    eroded = cv2.erode(obj_uint, kernel, iterations=1)
    boundary = obj_uint ^ eroded  # граница объекта

    boundary_vals = grad_mag[boundary > 0]
    if boundary_vals.size == 0:
        mean_grad = np.nan
        std_grad = np.nan
    else:
        mean_grad = float(boundary_vals.mean())
        std_grad = float(boundary_vals.std())

    # контраст объект–фон
    obj_vals = img_float[obj_mask]
    bg_vals = img_float[bg_mask]

    if obj_vals.size == 0 or bg_vals.size == 0:
        contrast_C = np.nan
    else:
        mu_obj = obj_vals.mean()
        mu_bg = bg_vals.mean()
        sigma_bg = bg_vals.std() + 1e-8
        contrast_C = float((mu_obj - mu_bg) / sigma_bg)

    return mean_grad, std_grad, contrast_C

2. Текстура внутри ооцита (GLCM)

In [7]:
def compute_glcm_features(img, mask, levels=32):
    """GLCM признаки внутри ооцита (по ROI по bounding-box).
       Возвращает: contrast, homogeneity, correlation, entropy
    """
    obj_mask, _ = get_object_background_masks(mask.astype(np.uint8))
    if obj_mask.sum() == 0:
        return np.nan, np.nan, np.nan, np.nan

    # bounding box маски
    ys, xs = np.where(obj_mask)
    y_min, y_max = ys.min(), ys.max()
    x_min, x_max = xs.min(), xs.max()

    roi = img[y_min:y_max+1, x_min:x_max+1]

    # квантуем яркость до levels уровней
    roi = roi.astype(np.float32)
    roi_norm = (roi - roi.min()) / (roi.max() - roi.min() + 1e-8)
    roi_q = (roi_norm * (levels - 1)).astype(np.uint8)

    # GLCM: расстояние 1 пиксель, угол 0
    glcm = graycomatrix(
        roi_q,
        distances=[1],
        angles=[0],
        levels=levels,
        symmetric=True,
        normed=True
    )

    contrast = float(graycoprops(glcm, 'contrast')[0, 0])
    homogeneity = float(graycoprops(glcm, 'homogeneity')[0, 0])
    correlation = float(graycoprops(glcm, 'correlation')[0, 0])

    # энтропия
    p = glcm.astype(np.float64)
    entropy = float(-np.sum(p * np.log2(p + 1e-12)))

    return contrast, homogeneity, correlation, entropy

3. SNR и доля энергии высоких частот

In [8]:
def compute_snr_and_highfreq_energy(img, mask):
    """SNR внутри ооцита и доля энергии высоких частот (по ROI).
       Возвращает: snr, high_freq_ratio
    """
    obj_mask, _ = get_object_background_masks(mask.astype(np.uint8))

    if obj_mask.sum() == 0:
        return np.nan, np.nan

    # SNR внутри объекта
    obj_vals = img[obj_mask].astype(np.float32)
    mu_obj = obj_vals.mean()
    sigma_obj = obj_vals.std() + 1e-8
    snr = float(mu_obj / sigma_obj)

    # FFT по ROI (bounding-box)
    ys, xs = np.where(obj_mask)
    y_min, y_max = ys.min(), ys.max()
    x_min, x_max = xs.min(), xs.max()

    roi = img[y_min:y_max+1, x_min:x_max+1].astype(np.float32)

    # вычитаем среднее для избежания DC-доминанты
    roi_zero = roi - roi.mean()

    F = np.fft.fft2(roi_zero)
    F_shift = np.fft.fftshift(F)
    power = np.abs(F_shift)**2

    h, w = roi.shape
    cy, cx = h // 2, w // 2

    Y, X = np.ogrid[:h, :w]
    dist = np.sqrt((Y - cy)**2 + (X - cx)**2)
    max_dist = dist.max() + 1e-8

    # высокочастотные компоненты: радиус > 0.5 * max
    high_mask = dist > 0.5 * max_dist

    total_energy = power.sum() + 1e-12
    high_energy = power[high_mask].sum()
    high_freq_ratio = float(high_energy / total_energy)

    return snr, high_freq_ratio

In [9]:
records = []

image_paths = sorted(
    [p for p in glob.glob(os.path.join(images_dir, "*")) if p.lower().endswith(".png")]
)

print(f"Найдено изображений: {len(image_paths)}")

for img_path in image_paths:
    img, mask = load_pair(img_path, masks_dir)
    fname = os.path.basename(img_path)

    if img is None or mask is None:
        print(f"Пропускаю {fname}: не удалось загрузить пару image+mask")
        continue

    mean_grad, std_grad, contrast_C = compute_boundary_gradient_and_contrast(img, mask)
    glcm_contrast, glcm_homog, glcm_corr, glcm_entropy = compute_glcm_features(img, mask)
    snr, high_freq_ratio = compute_snr_and_highfreq_energy(img, mask)

    records.append({
        "filename": fname,

        # Граница / контраст
        "mean_grad_boundary": mean_grad,
        "std_grad_boundary": std_grad,
        "contrast_obj_bg": contrast_C,

        # GLCM
        "glcm_contrast": glcm_contrast,
        "glcm_homogeneity": glcm_homog,
        "glcm_correlation": glcm_corr,
        "glcm_entropy": glcm_entropy,

        # Шум / высокие частоты
        "snr_obj": snr,
        "high_freq_energy_ratio": high_freq_ratio,
    })

df = pd.DataFrame(records)
df.head()

Найдено изображений: 600


,filename,mean_grad_boundary,std_grad_boundary,contrast_obj_bg,glcm_contrast,glcm_homogeneity,glcm_correlation,glcm_entropy,snr_obj,high_freq_energy_ratio
0,001_0002.png,36.197773,24.698957,-0.584804,0.382729,0.840217,0.982134,4.802470,6.862914,0.001299
1,001_0004.png,49.779606,50.088978,-0.638517,0.461631,0.820818,0.984781,5.022961,7.252433,0.001307
2,001_0013.png,29.767946,20.753317,-0.663486,0.367531,0.845727,0.981821,4.628230,6.351831,0.001762
3,001_0018.png,34.106888,22.655668,-0.604392,0.372840,0.843447,0.981544,4.733351,6.337750,0.001330
4,001_0019.png,43.135754,29.287651,-0.596329,0.355484,0.850820,0.981017,4.575270,6.902669,0.001273


In [10]:
numeric_df = df.select_dtypes(include=[np.number])
mean_metrics = numeric_df.mean()
std_metrics = numeric_df.std()
median_metrics = numeric_df.median()
min_metrics = numeric_df.min()
max_metrics = numeric_df.max()

summary_df = pd.DataFrame({
    "mean": mean_metrics,
    "std": std_metrics,
    "median": median_metrics,
    "min": min_metrics,
    "max": max_metrics
})

summary_df

,mean,std,median,min,max
mean_grad_boundary,78.974882,26.167347,75.333935,29.746655,190.082657
std_grad_boundary,52.441461,19.668939,49.740135,16.884005,144.421097
contrast_obj_bg,-1.291449,0.541304,-1.194337,-4.248947,-0.130100
glcm_contrast,0.518836,0.269505,0.444643,0.187995,2.094405
glcm_homogeneity,0.838959,0.045293,0.847190,0.613000,0.921045
glcm_correlation,0.984654,0.013673,0.989199,0.893495,0.997498
glcm_entropy,4.820002,0.495767,4.825786,3.429059,6.454231
snr_obj,6.042025,1.245043,5.854501,3.308457,12.908435
high_freq_energy_ratio,0.001538,0.001921,0.000815,0.000270,0.012289


### Граница и контраст (критерий 1)

**Градиент по границе**

* mean_grad_boundary ≈ 79 ± 26
* Разброс заметный (min ≈ 29, max ≈ 190)

* В среднем граница выражена достаточно хорошо, но высокая дисперсия говорит о неоднородности контуров по датасету.
* Есть кадры со слабым градиентом (~30) → в них Canny/морфология будут нестабильны.

**Контраст объект–фон**

* contrast_obj_bg ≈ -1.29 ± 0.54

Отрицательное значение означает, что ооцит темнее фона

* Контраст умеренный (по модулю ~1.3)
* Минимум ~ -4.25 → в некоторых кадрах контраст очень высокий
* Максимум ~ -0.13 → есть кадры почти без различия

Есть подмножество кадров с почти нулевым контрастом → пороговые и морфологические методы там должны деградировать.

### Текстурная сложность (критерий 2)

**GLCM Contrast ≈ 0.52**

Низко-среднее значение → текстура не контрастная.

**GLCM Homogeneity ≈ 0.84**

Высокая однородность → структура внутри в целом регулярная.

**GLCM Correlation ≈ 0.98**

Очень высокая корреляция → сильная пространственная зависимость пикселей.

**GLCM Entropy ≈ 4.82**

Средняя энтропия (max ~6.45) → присутствует вариативность текстуры.

* Ооциты **в целом достаточно однородны**, но присутствует умеренная текстурная вариабельность.
* Морфология должна работать на «простых» кадрах, но при росте энтропии/contrast GLCM → CNN получит преимущество.

### Шум и частотный анализ

**SNR ≈ 6.04 ± 1.25**

Это достаточно хорошее отношение сигнал/шум. Кадры с SNR ~3.3 — потенциально проблемные.

**High-frequency energy ratio ≈ 0.0015**

Очень маленькое значение.

* изображение в целом гладкое, нет выраженного высокочастотного шума.
* Проблемы морфологии не связаны с шумом, а скорее с вариабельностью границы и контраста.

In [14]:
def compute_boundary_stats(img, mask):
    img_float = img.astype(np.float32)

    # Градиенты Sobel
    gx = cv2.Sobel(img_float, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(img_float, cv2.CV_32F, 0, 1, ksize=3)
    grad_mag = np.sqrt(gx**2 + gy**2)

    # Бинарная маска ооцита
    obj = (mask > 0).astype(np.uint8)

    if obj.sum() == 0:
        # на всякий случай, если маска пустая
        return {
            "mean_grad_boundary": np.nan,
            "std_grad_boundary": np.nan,
            "cv_grad_boundary": np.nan,
            "min_grad_boundary": np.nan,
            "max_grad_boundary": np.nan,
            "p10_grad_boundary": np.nan,
            "p50_grad_boundary": np.nan,
            "p90_grad_boundary": np.nan,
        }

    # Граница: XOR маски и её эрозии
    kernel = np.ones((3, 3), np.uint8)
    eroded = cv2.erode(obj, kernel, iterations=1)
    boundary = obj ^ eroded

    boundary_vals = grad_mag[boundary > 0]

    if boundary_vals.size == 0:
        return {
            "mean_grad_boundary": np.nan,
            "std_grad_boundary": np.nan,
            "cv_grad_boundary": np.nan,
            "min_grad_boundary": np.nan,
            "max_grad_boundary": np.nan,
            "p10_grad_boundary": np.nan,
            "p50_grad_boundary": np.nan,
            "p90_grad_boundary": np.nan,
        }

    mean_grad = float(boundary_vals.mean())
    std_grad = float(boundary_vals.std())
    cv_grad = float(std_grad / (mean_grad + 1e-8))
    min_grad = float(boundary_vals.min())
    max_grad = float(boundary_vals.max())
    p10 = float(np.percentile(boundary_vals, 10))
    p50 = float(np.percentile(boundary_vals, 50))
    p90 = float(np.percentile(boundary_vals, 90))

    return {
        "mean_grad_boundary": mean_grad,
        "std_grad_boundary": std_grad,
        "cv_grad_boundary": cv_grad,
        "min_grad_boundary": min_grad,
        "max_grad_boundary": max_grad,
        "p10_grad_boundary": p10,
        "p50_grad_boundary": p50,
        "p90_grad_boundary": p90,
    }

In [15]:
def load_pair(image_path, masks_dir):
    fname = os.path.basename(image_path)
    mask_path = os.path.join(masks_dir, fname)
    if not os.path.exists(mask_path):
        return None, None
    img  = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    return img, mask

In [16]:
image_paths = sorted(
    [p for p in glob.glob(os.path.join(images_dir, "*")) if p.lower().endswith(".png")]
)

boundary_records = []

print(f"Найдено изображений: {len(image_paths)}")

for img_path in image_paths:
    fname = os.path.basename(img_path)
    img, mask = load_pair(img_path, masks_dir)

    if img is None or mask is None:
        print(f"Пропускаю {fname}: не удалось загрузить пару image+mask")
        continue

    stats = compute_boundary_stats(img, mask)
    stats["filename"] = fname
    boundary_records.append(stats)

df_boundary = pd.DataFrame(boundary_records)
df_boundary.head()

Найдено изображений: 600


,mean_grad_boundary,std_grad_boundary,cv_grad_boundary,min_grad_boundary,max_grad_boundary,p10_grad_boundary,p50_grad_boundary,p90_grad_boundary,filename
0,36.197773,24.698957,0.682334,0.0,117.004272,9.486833,30.149513,74.018913,001_0002.png
1,49.779606,50.088978,1.006215,0.0,239.169403,9.486833,33.674866,132.766678,001_0004.png
2,29.767946,20.753317,0.697170,0.0,116.764717,7.071068,25.059929,57.428215,001_0013.png
3,34.106888,22.655668,0.664255,0.0,163.229904,8.944272,29.832869,65.391129,001_0018.png
4,43.135754,29.287651,0.678965,0.0,165.239227,11.045361,36.715118,87.783829,001_0019.png


In [17]:
# Оставляем только числовые столбцы
numeric_boundary = df_boundary.select_dtypes(include=[np.number])

# Среднее, std, медиана, минимум и максимум по каждой метрике
boundary_summary = pd.DataFrame({
    "mean":   numeric_boundary.mean(),
    "std":    numeric_boundary.std(),
    "median": numeric_boundary.median(),
    "min":    numeric_boundary.min(),
    "max":    numeric_boundary.max(),
})

boundary_summary

,mean,std,median,min,max
mean_grad_boundary,78.974882,26.167347,75.333935,29.746655,190.082657
std_grad_boundary,52.441461,19.668939,49.740135,16.884005,144.421097
cv_grad_boundary,0.666176,0.099788,0.664173,0.366909,1.006215
min_grad_boundary,0.469785,0.808533,0.000000,0.000000,5.099020
max_grad_boundary,286.542316,106.998921,272.877258,103.768974,734.024536
p10_grad_boundary,19.579525,8.250957,17.944100,4.472136,59.074528
p50_grad_boundary,69.470313,23.344470,66.121101,23.537205,163.113464
p90_grad_boundary,152.346406,54.649000,143.092628,52.000000,452.044250


**Средний уровень градиента**

mean_grad_boundary ≈ 79 ± 26

* Граница в среднем выражена достаточно хорошо.
* Но стандартное отклонение 26 — это **≈33% от среднего**.
* Минимум ≈ 29 → есть кадры со слабой границей.
* Максимум ≈ 190 → есть кадры с очень резкой границей.
* Контраст контура существенно варьирует между изображениями.


**Насколько “скачет” градиент внутри одного изображения**


**cv_grad_boundary ≈ 0.666 ± 0.099**

Коэффициент вариации ~0.67 - это очень высокая неоднородность.

* CV < 0.2 → граница почти равномерная
* CV ~ 0.3–0.4 → умеренная неоднородность
* CV > 0.6 → граница сильно неравномерная

Среднее ~0.66 и максимум >1.0.

Максимум CV ≈ 1.00

Граница в этих кадрах экстремально нестабильная: местами резкая, местами почти отсутствует.

**Квантильный анализ (внутри одного изображения)**

p10 ≈ 19.6

p50 ≈ 69.5

p90 ≈ 152.3

В типичном кадре:

* 10% границы имеют градиент < 20
* 50% — около 70
* 10% — выше 150

* То есть разброс почти в 8 раз (20 → 150).
* В одном и том же контуре есть участки практически “размытые” и участки очень резкие.

**Минимальные значения градиента**

`min_grad_boundary` median ≈ 0
минимум = 0

* в большинстве изображений есть участки границы, где градиент практически отсутствует.
* Морфологические и пороговые методы будут “разрываться” именно в этих местах.


**Максимальные значения**

`max_grad_boundary` mean ≈ 286
max ≈ 734

Это говорит о локальных очень резких переходах — возможно: полярное тельце, участки пересвета, артефакты освещения.